In [1]:
import sys
import numpy as np
import pandas as pd
import gamspy as gp

from src.parameters import *

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container(options=gp.Options(relative_optimality_gap=0))

In [2]:
# LOAD DATA
teams_df = pd.read_csv("data/processed/hq.csv")
circuits_df = pd.read_csv("data/processed/circuits.csv")
distances_df = pd.read_csv("data/processed/distances.csv")
climate_df = pd.read_csv("data/processed/climate.csv")
festivals_df = pd.read_csv("data/processed/festivals.csv")

In [3]:
rev_df = distances_df.rename(columns={"from": "to", "to": "from"})

distances_sym_df = (
    pd.concat([distances_df, rev_df], ignore_index=True)
      .drop_duplicates(subset=["from", "to"])
)

In [4]:
festivals_df

,circuit_id,week_num
0,ABD,1
1,ABD,2
2,ABD,3
3,ABD,4
4,SIN,24
5,JPN,10
6,CAN,1
7,ESP,30
8,MEX,36
9,USA_COT,3


In [5]:
# CONSTANTS
BREAK_START_DATE = 24 # 2026-08-09 (in climate_df)
BREAK_END_DATE = 26   # 2026-08-30 (in climate_df) 
BREAK_POINT = 14      # NO. OF RACES BEFROE BREAK

# PARAMETERS identifying AUS and ABD
FIRST = "AUS"
LAST = "ABD"

# Identify last weekend before break and first after break
t_pre  = str(BREAK_START_DATE - 1)
t_post = str(BREAK_END_DATE   + 1)

MAX_TRIPLE_HEADERS = 3

In [6]:
# PREPROCESSING

teams = teams_df["team_id"].tolist()[:-1]
circuits = circuits_df["circuit_id"].tolist()

weekends = climate_df['week_num'].astype(int).tolist()
summer_break = climate_df.loc[
    climate_df['week_num'].between(BREAK_START_DATE, BREAK_END_DATE), 
    'week_num'].astype(int).tolist()


race_emissions = distances_sym_df[distances_sym_df['from'].isin(circuits) & 
                                  distances_sym_df['to'].isin(circuits)][["from", "to", "emissions_kgCO2e"]]
hq_emissions = distances_sym_df[distances_sym_df['from'].isin(teams)][["from", "to", "emissions_kgCO2e"]]

In [7]:
n_teams = len(teams)
n_races = len(circuits)
n_weekends = len(weekends)
team_ratio = 1

In [8]:
feasible_dates = np.zeros((n_weekends, n_races))
feasible_dates_df = pd.DataFrame(feasible_dates, columns=circuits, index=climate_df['week_num'].astype(int).tolist())

climate_df.index = climate_df['week_num'].astype(int).tolist()

for circuit in circuits:
    temp_col = f"{circuit}_avg_temp"
    precip_col = f"{circuit}_avg_precip"

    ok_temp = climate_df[temp_col].between(MIN_TEMP_F, MAX_TEMP_F)
    ok_precip = climate_df[precip_col] < MAX_PRECIP_IN

    feasible_dates_df.loc[ok_temp & ok_precip, circuit] = 1

# Festival filter
for idx, row in festivals_df.iterrows():
    feasible_dates_df.iloc[row['week_num'] -1 , feasible_dates_df.columns.get_loc(row['circuit_id'])] = 0

In [9]:
# Sentivity analysis:

# for i in circuits:
#     for j in circuits:
#         if i != j:
#             FIRST = i
#             LAST = j

In [10]:
# SET
Weekend = gp.Set(m, records=weekends)
SummerBreak = gp.Set(m, domain=[Weekend], records=summer_break)
Team = gp.Set(m, records=teams)
Circuit = gp.Set(m, records=circuits)
i = gp.Alias(m, alias_with=Circuit)
j = gp.Alias(m, alias_with=Circuit)
t = gp.Alias(m, alias_with=Weekend)



# PARAMETERS
r_emissions = gp.Parameter(m, domain=[Circuit, Circuit], records=race_emissions,
                         description="emissions between races in kgCO2e")
hq_emissions = gp.Parameter(m, domain=[Team, Circuit], records=hq_emissions,
                         description="emissions from HQ to race in kgCO2e")

feasible_dates = gp.Parameter(m, domain=[Weekend, Circuit], records=feasible_dates_df.stack().reset_index().values.tolist(),
                            description="feasibility of holding race on weekend (1 if feasible, 0 otherwise)")
feasible_dates[SummerBreak, Circuit] = 0 # No races during the break
    

first_set = gp.Parameter(m, records=14,
                         description='Number of races before the summer break')
break_start_date = gp.Parameter(m, records=np.array(BREAK_START_DATE),
                                description='Weekend where break begins')
break_end_date = gp.Parameter(m, records=np.array(BREAK_END_DATE),
                                description='Weekend where break ends')

In [11]:
# VARIABLES
x = gp.Variable(m,type='binary',domain=[Circuit,Circuit],
                description="1 if race in i is scheduled immediately before j, 0 otherwise")
y = gp.Variable(m, type='binary', domain=[Circuit,Weekend], 
                description='1 if race in Circuit is scheduled on Weekend, 0 otherwise')
z = gp.Variable(m, type="binary", domain=[i, j],
                description="1 if race i is on pre-break weekend and race j on post-break weekend")
u = gp.Variable(m,type='positive',domain=[Circuit], 
                description="Position of race in the calendar sequence")
# Binary variable: 1 if weeks t, t+1, t+2 all have races (triple header)
triple_header = gp.Variable(m, type='binary', domain=[t],
                            description="1 if week t starts a triple header")


x.fx[i,i] = 0

u.lo[Circuit] = 2                   # all races but first must be at least position 2
u.up[Circuit] = gp.Card(Circuit)    # all races must be at most position number of races

u.fx[FIRST] = 1                # First race = AUS
u.fx[LAST] = gp.Card(Circuit)  # Last race = ABD (position = number of races)



In [12]:
# EQUATIONS
# Obvious ones first
assign1 = gp.Equation(m, domain=[j],
                      description='Each circuit has exactly one predecessor')
assign1[j] = gp.Sum(i, x[i,j]) == 1

assign2 = gp.Equation(m, domain=[i],
                      description='Each circuit has exactly one successor')
assign2[i] = gp.Sum(j, x[i,j]) == 1

assign3 = gp.Equation(m, domain=[i],
                      description='Each race is held exactly once')
assign3[i] = gp.Sum(t, y[i,t]) == 1

assign4 = gp.Equation(m, domain=[t],
                      description='Each weekend has at most one race')
assign4[t] = gp.Sum(i, y[i,t]) <= 1

# Get your actual MTZ ordinal positions
FIRST_ord = circuits.index(FIRST) + 1
LAST_ord = circuits.index(LAST) + 1

print(f"{FIRST} is at ordinal position: {FIRST_ord}")
print(f"{LAST} is at ordinal position: {LAST_ord}")

# Then use ordinals instead of strings
mtz = gp.Equation(m, domain=[i,j])
mtz[i,j].where[(i.ord != FIRST_ord) & (j.ord != FIRST_ord) & (i.ord != LAST_ord)] = (
    u[i] - u[j] + 1 <= (gp.Card(Circuit) - 1) * (1 - x[i,j])
)

AUS is at ordinal position: 3
ABD is at ordinal position: 24


In [13]:
# Time Constraints
feasibiliy = gp.Equation(m, domain=[i,t],
                         description='Race can be held only if the weekend is feasible')
feasibiliy[i,t] = y[i,t] <= feasible_dates[t,i]


time_link = gp.Equation(m, domain=[i,j])
time_link[i,j].where[(i.ord != LAST_ord) | (j.ord != FIRST_ord)] = (
    gp.Sum(t, t.ord * y[j,t]) >= gp.Sum(t, t.ord * y[i,t]) + x[i,j] - (1 - x[i,j]) * gp.Card(Weekend)
)

triple_header_def = gp.Equation(m, domain=[t],
                                description='Calculating triple headers')
triple_header_def[t].where[t.ord <= gp.Card(Weekend) - 2] = (
    triple_header[t] * 3 <= gp.Sum(i, y[i, t]) + gp.Sum(i, y[i, t.lead(1)]) + gp.Sum(i, y[i, t.lead(2)])
)

triple_header_limit = gp.Equation(m,
                                  description='maximum triple headers constraint')
triple_header_limit[...] = gp.Sum(t, triple_header[t]) <= MAX_TRIPLE_HEADERS

break_partition = gp.Equation(m,
                              description='Exactly 14 races before summer break')
break_partition[...] = gp.Sum([i, t], y[i,t].where[t.ord < BREAK_START_DATE]) == first_set

pre_break_race = gp.Equation(m,
                             description='Exactly one race on last weekend before break')
pre_break_race[...] = gp.Sum(i, y[i, str(BREAK_START_DATE - 1)]) == 1

post_break_race = gp.Equation(m,
                              description='Exactly one race on first weekend after break')
post_break_race[...] = gp.Sum(i, y[i, str(BREAK_END_DATE + 1)]) == 1

In [14]:
z_lin1 = gp.Equation(m, domain=[i, j])
z_lin1[i, j] = z[i, j] <= x[i, j]

z_lin2 = gp.Equation(m, domain=[i, j])
z_lin2[i, j] = z[i, j] <= y[i, t_pre]

z_lin3 = gp.Equation(m, domain=[i, j])
z_lin3[i, j] = z[i, j] <= y[j, t_post]

z_lin4 = gp.Equation(m, domain=[i, j])
z_lin4[i, j] = z[i, j] >= x[i, j] + y[i, t_pre] + y[j, t_post] - 2


In [15]:
# Objective (your original was almost correct!):
total_emissions = (
    # 1. HQ -> FIRST RACE
    gp.Sum(Team, hq_emissions[Team, FIRST]) * team_ratio
    
    # 2. Sum of all consecutive race emissions
    + gp.Sum([i, j], r_emissions[i, j] * x[i, j]) * 10
    
    # 3. Last race -> HQ
    + gp.Sum(Team, hq_emissions[Team, LAST]) * team_ratio

    # ---- SUMMER BREAK ADJUSTMENTS ----
    
    # 4. REMOVE emission of break connection
    - gp.Sum([i, j], r_emissions[i, j] * z[i, j]) * 10
    
    # 5. ADD: pre-break race -> HQ
    + gp.Sum([Team, i], hq_emissions[Team, i] * y[i, t_pre]) * team_ratio
    
    # 6. ADD: HQ -> post-break race
    + gp.Sum([Team, j], hq_emissions[Team, j] * y[j, t_post]) * team_ratio

    # ---- LOOP CLOSURE ADJUSTMENT ----
    
    # 7. REMOVE LAST RACE -> FIRST RACE
    - r_emissions[LAST, FIRST] * x[LAST, FIRST] * 10
)

model = gp.Model(
    m,
    name="f1_calendar",
    equations=m.getEquations(),
    sense=gp.Sense.MIN,
    problem=gp.Problem.MIP,
    objective=total_emissions
)

model.solve(solver="gurobi", options=gp.Options(
    time_limit=300,           # 5 minutes
    relative_optimality_gap=0.05
))

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,5.114427e+06,4526,2201,MIP,GUROBI,13.836


In [16]:
df = x.records                  # adjacency matrix in long form
df2 = u.records                 # each race with its order (u variable)
df1 = df[df['level'] == 1]      # edges actually used (x[i,j] = 1)


In [17]:
import plotly.graph_objects as go
import numpy as np

# Step 1: Race order
order_df = df2.sort_values("level").reset_index(drop=True)

# Step 2: Edges
edges = df1[['Circuit_0', 'Circuit_1']].reset_index(drop=True)

# Step 3: Coordinates
coords = circuits_df[['circuit_id', 'latitude', 'longitude']]

edges = (
    edges.merge(coords, left_on='Circuit_0', right_on='circuit_id')
         .rename(columns={'latitude':'lat_from', 'longitude':'lon_from'})
         .drop(columns=['circuit_id'])
         .merge(coords, left_on='Circuit_1', right_on='circuit_id')
         .rename(columns={'latitude':'lat_to', 'longitude':'lon_to'})
         .drop(columns=['circuit_id'])
)

fig = go.Figure()

# Function to create arrowhead
def arrowhead(lat_from, lon_from, lat_to, lon_to, scale=0.15):
    """Return a small arrowhead point slightly before the destination."""

    # Direction vector
    dlat = lat_to - lat_from
    dlon = lon_to - lon_from

    # Shorten vector for arrowhead base
    lat_arrow = lat_to - scale * dlat
    lon_arrow = lon_to - scale * dlon

    return lat_arrow, lon_arrow

# Draw route lines + arrowheads
for _, row in edges.iterrows():

    # Main route line
    fig.add_trace(go.Scattergeo(
        lon=[row['lon_from'], row['lon_to']],
        lat=[row['lat_from'], row['lat_to']],
        mode='lines',
        line=dict(width=2, color='red'),
        opacity=0.8,
        name=f"{row['Circuit_0']} → {row['Circuit_1']}"
    ))

    # Arrowhead segment
    lat_arrow, lon_arrow = arrowhead(row['lat_from'], row['lon_from'],
                                     row['lat_to'], row['lon_to'])

    fig.add_trace(go.Scattergeo(
        lon=[lon_arrow, row['lon_to']],
        lat=[lat_arrow, row['lat_to']],
        mode='lines',
        line=dict(width=4, color='red'),
        opacity=1.0,
        showlegend=False
    ))

# Circuit nodes
fig.add_trace(go.Scattergeo(
    lon=coords['longitude'],
    lat=coords['latitude'],
    mode='markers+text',
    marker=dict(size=8, color="blue"),
    text=coords['circuit_id'],
    textposition="top center",
    name="Circuits"
))

fig.update_layout(
    title="F1 Calendar Travel Route (with Direction Arrows)",
    geo=dict(
        projection_type="natural earth",
        showcountries=True,
        landcolor="rgb(240, 240, 240)",
    ),
    height=650,
    showlegend=False 
)

fig.show()


In [18]:
pd.merge(df1, df2, left_on = 'Circuit_0', right_on = 'Circuit').sort_values(by='level_y')[['Circuit_0', 'Circuit_1', 'level_y']]

,Circuit_0,Circuit_1,level_y
2,AUS,SIN,1.0
16,SIN,CHN,2.0
4,CHN,JPN,3.0
3,JPN,USA_LVG,4.0
19,USA_LVG,CAN,5.0
7,CAN,MEX,6.0
20,MEX,USA_COT,7.0
18,USA_COT,MIA,8.0
5,MIA,BRA,9.0
21,BRA,ESP,10.0


In [19]:
df = y.records
df = df[df['level'] == 1][["Circuit", "Weekend"]]
df.sort_values(by='Weekend')

,Circuit,Weekend
83,AUS,2
658,SIN,3
167,CHN,4
127,JPN,5
784,USA_LVG,6
293,CAN,7
827,MEX,8
746,USA_COT,9
214,MIA,10
874,BRA,14
